# Gibbs Sampler for Gaussian Mixtures

## Introduction

The Gibbs sampler is a special case of MCMC where we update one coordinate at a time using its conditional distribution. This notebook demonstrates:

1. **Gibbs sampling fundamentals** - Updating coordinates sequentially
2. **Metropolis-within-Gibbs** - A hybrid approach for when conditionals are not available
3. **Comparison with Metropolis-Hastings** - Understanding when Gibbs is more efficient

## Setup

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from src.gibbs_sampler import GibbsSampler
from src.metropolis_hastings import MetropolisHastings
from src.target_distributions import mixture_of_gaussians_log_pdf, get_distribution
from src.utils import compute_effective_sample_size, plot_contour, plot_autocorrelation
import seaborn as sns
sns.set_style('whitegrid')

%matplotlib inline

## 1. Target Distribution

We'll use the same mixture of Gaussians as before for easy comparison.

In [ ]:
# Visualize the target
x = np.linspace(-6, 6, 100)
y = np.linspace(-6, 6, 100)
X, Y = np.meshgrid(x, y)
grid_points = np.column_stack([X.ravel(), Y.ravel()])

log_pdf_values = np.array([mixture_of_gaussians_log_pdf(p) for p in grid_points])
pdf_values = np.exp(log_pdf_values - np.max(log_pdf_values))
pdf_grid = pdf_values.reshape(X.shape)

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contourf(X, Y, pdf_grid, levels=50, cmap='viridis')
ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
ax.set_title('Target Distribution: Mixture of Gaussians')
plt.colorbar(contour, label='Density')
plt.show()

## 2. Running the Gibbs Sampler

The Gibbs sampler updates each coordinate sequentially. Here we use a Metropolis-within-Gibbs approach where each coordinate update uses a Metropolis acceptance step.

In [ ]:
# Initialize the Gibbs sampler
gibbs_sampler = GibbsSampler(
    target_log_pdf=mixture_of_gaussians_log_pdf,
    n_dim=2
)

# Run sampling
gibbs_samples, gibbs_diagnostics = gibbs_sampler.sample(
    n_samples=10000,
    burn_in=2000,
    thin=5
)

print(f"Acceptance rate (Metropolis-within-Gibbs): {gibbs_diagnostics['acceptance_rate']:.3f}")
print(f"Number of samples: {len(gibbs_samples)}")

## 3. Visualizing Gibbs Results

### 3.1 Trace Plots

In [ ]:
fig = gibbs_sampler.plot_chain()
plt.savefig('../results/figures/gibbs_trace_plots.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.2 Contour Plot of Samples

In [ ]:
fig = plot_contour(
    mixture_of_gaussians_log_pdf,
    gibbs_samples,
    title='Gibbs Sampler Samples on Mixture of Gaussians',
    save_path='../results/figures/gibbs_samples_contour.png'
)
plt.show()

### 3.3 Autocorrelation Analysis

In [ ]:
fig = plot_autocorrelation(gibbs_samples, max_lag=100)
plt.savefig('../results/figures/gibbs_autocorrelation.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.4 Convergence Diagnostics

In [ ]:
# Effective Sample Size
gibbs_ess = compute_effective_sample_size(gibbs_samples)
print(f"Gibbs Effective Sample Size: {gibbs_ess:.0f} (out of {len(gibbs_samples)})")
print(f"ESS / Total samples: {gibbs_ess/len(gibbs_samples):.2%}")

if gibbs_ess > 0.5 * len(gibbs_samples):
    print("✅ Good mixing: ESS > 50% of total samples")
else:
    print("⚠️ Poor mixing: ESS < 50% of total samples")

# Run Metropolis-Hastings for comparison
mh_sampler = MetropolisHastings(
    target_log_pdf=mixture_of_gaussians_log_pdf,
    proposal_scale=0.5,
    n_dim=2,
    adapt_scale=True
)

mh_samples, mh_diagnostics = mh_sampler.sample(
    n_samples=10000,
    burn_in=2000,
    thin=5,
    progress_bar=False
)

mh_ess = compute_effective_sample_size(mh_samples)

print("Comparison Summary:")
print(f"MH Acceptance Rate: {mh_diagnostics['acceptance_rate']:.3f}")
print(f"Gibbs Acceptance Rate: {gibbs_diagnostics['acceptance_rate']:.3f}")
print(f"MH ESS: {mh_ess:.0f}")
print(f"Gibbs ESS: {gibbs_ess:.0f}")
print(f"MH Efficiency: {mh_ess/len(mh_samples):.2%}")
print(f"Gibbs Efficiency: {gibbs_ess/len(gibbs_samples):.2%}")

### 4.1 Side-by-Side Sample Comparison

In [ ]:
# Create side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot MH samples
axes[0].contourf(X, Y, pdf_grid, levels=20, alpha=0.3, cmap='viridis')
axes[0].scatter(mh_samples[:, 0], mh_samples[:, 1], alpha=0.05, s=3, c='red')
axes[0].set_title(f'Metropolis-Hastings\nESS = {mh_ess:.0f}')
axes[0].set_xlabel('Dimension 1')
axes[0].set_ylabel('Dimension 2')

# Plot Gibbs samples
axes[1].contourf(X, Y, pdf_grid, levels=20, alpha=0.3, cmap='viridis')
axes[1].scatter(gibbs_samples[:, 0], gibbs_samples[:, 1], alpha=0.05, s=3, c='blue')
axes[1].set_title(f'Gibbs Sampler\nESS = {gibbs_ess:.0f}')
axes[1].set_xlabel('Dimension 1')
axes[1].set_ylabel('Dimension 2')

plt.tight_layout()
plt.savefig('../results/figures/gibbs_vs_mh_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.2 Autocorrelation Comparison

Gibbs typically has lower autocorrelation because it updates one coordinate at a time.

In [ ]:
from src.utils import compute_autocorrelation

# Compute autocorrelation for both samplers
max_lag = 50
mh_autocorr = compute_autocorrelation(mh_samples, max_lag)
gibbs_autocorr = compute_autocorrelation(gibbs_samples, max_lag)

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Dimension 1
ax1.plot(mh_autocorr[:, 0], label='MH', alpha=0.7)
ax1.plot(gibbs_autocorr[:, 0], label='Gibbs', alpha=0.7)
ax1.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax1.set_xlabel('Lag')
ax1.set_ylabel('Autocorrelation')
ax1.set_title('Dimension 1 Autocorrelation')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dimension 2
ax2.plot(mh_autocorr[:, 1], label='MH', alpha=0.7)
ax2.plot(gibbs_autocorr[:, 1], label='Gibbs', alpha=0.7)
ax2.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax2.set_xlabel('Lag')
ax2.set_ylabel('Autocorrelation')
ax2.set_title('Dimension 2 Autocorrelation')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/gibbs_vs_mh_autocorrelation.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Mode Switching Analysis

Let's check if Gibbs samples the modes in the correct proportions.

In [ ]:
# Define mode centers
mode1_center = np.array([-3, -3])
mode2_center = np.array([3, 3])

# Assign Gibbs samples to nearest mode
dist_to_mode1 = np.linalg.norm(gibbs_samples - mode1_center, axis=1)
dist_to_mode2 = np.linalg.norm(gibbs_samples - mode2_center, axis=1)
mode_assignment = np.argmin(np.column_stack([dist_to_mode1, dist_to_mode2]), axis=1)

# Count mode visits
mode_counts = np.bincount(mode_assignment)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Mode switching
ax1.plot(mode_assignment[:500], 'o-', markersize=3)
ax1.set_ylabel('Mode Assignment')
ax1.set_xlabel('Iteration')
ax1.set_title('Gibbs Mode Switching (First 500 samples)')
ax1.set_yticks([0, 1])
ax1.set_yticklabels(['Mode 1 (-3,-3)', 'Mode 2 (3,3)'])

# Mode proportions
ax2.bar(['Mode 1', 'Mode 2'], mode_counts)
ax2.set_ylabel('Count')
ax2.set_title(f'Mode Visits (Ratio: {mode_counts[0]/mode_counts[1]:.2f})')

plt.tight_layout()
plt.savefig('../results/figures/gibbs_mode_switching.png', dpi=300, bbox_inches='tight')
plt.show()

# Check against expected ratio
expected_ratio = 0.4 / 0.6
actual_ratio = mode_counts[0] / mode_counts[1]
print(f"Expected mode ratio: {expected_ratio:.3f}")
print(f"Actual mode ratio: {actual_ratio:.3f}")
if abs(actual_ratio - expected_ratio) < 0.1:
    print("✅ Gibbs explores both modes proportionally!")
else:
    print("⚠️ Gibbs may be struggling to switch modes")

## 6. Summary Statistics

In [ ]:
stats = gibbs_sampler.get_summary_statistics()
print("Gibbs Sampler Summary Statistics:")
for dim, values in stats.items():
    print(f"\n{dim}:")
    print(f"  Mean: {values['mean']:.3f}")
    print(f"  Std: {values['std']:.3f}")
    print(f"  50% (median): {values['quantiles']['50%']:.3f}")
    print(f"  95% CI: [{values['quantiles']['2.5%']:.3f}, {values['quantiles']['97.5%']:.3f}]")

## 7. Conclusion

**Key Takeaways:**

1. **Gibbs sampling advantages**:
   - Often has lower autocorrelation than MH
   - Can be more efficient for high-dimensional problems
   - Updates one coordinate at a time, which can help with correlated distributions

2. **Gibbs sampling limitations**:
   - Requires conditional distributions (or approximations)
   - Can still get stuck in multi-modal distributions
   - May take longer to mix if coordinates are highly correlated

3. **Metropolis-within-Gibbs**:
   - A practical hybrid approach
   - Makes Gibbs applicable to more distributions
   - Introduces some rejection like MH

**Next Steps:**
- Benchmark against Hamiltonian Monte Carlo (notebook 3)
- Try on more challenging distributions
- Implement exact Gibbs for Gaussian cases